# Lezione 1 — Introduzione all'IA avanzata e la tua prima rete neurale

**Corso: Advanced Deep Learning**

## Obiettivi di questa lezione

- Collochi il deep learning avanzato (*deep learning*, DL) nel panorama dell'intelligenza artificiale (IA) e conosci le architetture principali che studierai nel corso
- Costruisci con NumPy il tuo primo neurone artificiale e un intero livello
- Calcoli a mano il gradiente di una rete minima e verifichi il risultato con PyTorch e il suo motore di differenziazione automatica (*autograd*)
- Definisci, addestri e valuti il tuo primo perceptrone multistrato (MLP) in PyTorch su un problema non lineare
- Esci da questa lezione con il codice di base che riutilizzerai durante il resto del corso

**Acronimi di questa lezione:** IA (intelligenza artificiale) · DL (*deep learning*) · ANN (rete neurale artificiale) · MLP (perceptrone multistrato) · CNN (rete neurale convoluzionale) · RNN (rete neurale ricorrente) · GAN (rete generativa avversaria) · VAE (autoencoder variazionale) · NLP (elaborazione del linguaggio naturale) · SGD (discesa del gradiente stocastica)

## La mappa del corso — che cos'è il deep learning avanzato?

- Il *deep learning* è un sottocampo del *machine learning* che usa reti neurali con molti livelli per estrarre pattern direttamente dai dati grezzi, senza progettare a mano le caratteristiche di input
- "Profondo" si riferisce al numero di livelli sovrapposti, non a una particolare sofisticatezza concettuale
- Casi reali: AlphaZero impara a giocare a scacchi giocando contro se stesso; AlphaFold prevede la struttura tridimensionale delle proteine con una precisione vicina ai metodi sperimentali

### Architetture che vedrai nel corso

| Architettura | Che cosa risolve | Idea chiave |
|---|---|---|
| Reti convoluzionali (CNN) | Dati con struttura spaziale (immagini) | Filtri locali riutilizzabili |
| Reti ricorrenti (RNN) | Dati sequenziali (testo, serie temporali) | Stato nascosto che ricorda il passato |
| Attenzione / Transformer | Dipendenze tra elementi di una sequenza | La rete "sceglie" su cosa concentrarsi |
| Reti generative avversarie (GAN) | Creare dati nuovi e realistici | Generatore contro discriminatore |
| Autoencoder variazionali (VAE) | Apprendere variabili latenti | Comprimere e ricostruire |
| IA responsabile | Robustezza, bias, impatto sociale | Parte integrante, non un'aggiunta |

- Oggi costruisci il pezzo comune a tutte queste architetture: il neurone artificiale e il suo addestramento tramite retropropagazione (*backpropagation*)

## Il tuo primo neurone

- Un neurone artificiale riceve degli input, li moltiplica per dei pesi che riflettono la loro importanza, aggiunge un termine di bias (*bias*) e fa passare il risultato attraverso una funzione di attivazione
- Formula: `z = w·x + b` ; `uscita = f(z)`
- Senza una funzione di attivazione non lineare, sovrapporre livelli non serve a nulla: l'insieme resterebbe una trasformazione affine
- Usiamo la sigmoide, che comprime qualsiasi valore nell'intervallo (zero, uno) e si interpreta come una probabilità

In [ ]:
import numpy as np

def sigmoide(z):
    """Comprime cualquier valor real en el rango (0, 1)."""
    return 1 / (1 + np.exp(-z))

def neurona(x, w, b):
    z = np.dot(x, w) + b   # suma ponderada
    return sigmoide(z)

# Ejemplo: ¿aprobamos un préstamo?
# x1 = ingresos (0=bajo, 1=alto), x2 = historial de pagos (0=malo, 1=excelente)
x = np.array([0.8, 0.9])
w = np.array([0.4, 1.2])   # el historial pesa más que los ingresos
b = -0.9

salida = neurona(x, w, b)
print(f"Salida: {salida:.3f} -> {'aprobado' if salida > 0.5 else 'rechazado'}")

# Perfil de riesgo
x_riesgo = np.array([0.2, 0.3])
print(f"Salida (riesgo): {neurona(x_riesgo, w, b):.3f}")

In [ ]:
# Una capa es simplemente varias neuronas en paralelo:
# cada fila de la matriz W son los pesos de una neurona distinta.
def capa(x, W, b):
    z = W @ x + b           # producto matricial + sesgos
    return sigmoide(z)

W_capa = np.array([
    [0.4, 1.2],    # neurona 1 (igual que antes)
    [1.0, -0.5],   # neurona 2 (prioriza ingresos altos)
])
b_capa = np.array([-0.9, -0.1])

print("Salidas de la capa:", capa(x, W_capa, b_capa))

## Backpropagation a mano

- La rete impara regolando i pesi per ridurre una funzione di perdita (*loss*)
- La *backpropagation* applica la regola della catena per calcolare quanto ciascun peso contribuisce all'errore finale
- Esempio minimo: un neurone con un solo peso `w`, senza bias, con attivazione sigmoide e perdita quadratica
- `ŷ = σ(w·x)` ; `perdita = (ŷ − y_obiettivo)²`
- Il gradiente si scompone in tre derivate concatenate: perdita→ŷ, ŷ→z, z→w

In [ ]:
x = 2.0
y_objetivo = 1.0
w = 0.3       # peso inicial (normalmente sería aleatorio)
eta = 0.1     # tasa de aprendizaje

def derivada_sigmoide(z):
    s = sigmoide(z)
    return s * (1 - s)   # propiedad de la sigmoide

for paso in range(5):
    # --- pase hacia adelante ---
    z = w * x
    y_pred = sigmoide(z)
    perdida = (y_pred - y_objetivo) ** 2

    # --- pase hacia atrás: regla de la cadena, término a término ---
    dL_dypred = 2 * (y_pred - y_objetivo)
    dypred_dz = derivada_sigmoide(z)
    dz_dw = x
    gradiente_w = dL_dypred * dypred_dz * dz_dw

    # --- descenso de gradiente ---
    w = w - eta * gradiente_w

    print(f"paso {paso}: w={w:.5f}  y_pred={y_pred:.5f}  "
          f"perdida={perdida:.5f}  gradiente={gradiente_w:.5f}")

## PyTorch e autograd

- PyTorch calcola i gradienti al posto tuo con il suo motore di differenziazione automatica, *autograd*
- `requires_grad=True` segna un tensore come parametro addestrabile
- `perdita.backward()` percorre il grafo computazionale all'indietro, esattamente come hai fatto a mano
- L'ottimizzatore applica la formula della discesa del gradiente al posto tuo
- Confronta i numeri con la cella precedente: devono coincidere passo per passo

In [ ]:
import torch

# Mismos datos que en la versión a mano
x_t = torch.tensor(2.0)
y_objetivo_t = torch.tensor(1.0)

# requires_grad=True: PyTorch registra las operaciones para poder derivarlas después
w_t = torch.tensor(0.3, requires_grad=True)

eta = 0.1
optimizador = torch.optim.SGD([w_t], lr=eta)

for paso in range(5):
    # --- pase hacia adelante ---
    z = w_t * x_t
    y_pred = torch.sigmoid(z)
    perdida = (y_pred - y_objetivo_t) ** 2

    # --- pase hacia atrás: autograd calcula dL/dw automáticamente ---
    optimizador.zero_grad()
    perdida.backward()

    print(f"paso {paso}: w={w_t.item():.5f}  y_pred={y_pred.item():.5f}  "
          f"perdida={perdida.item():.5f}  gradiente={w_t.grad.item():.5f}")

    # --- descenso de gradiente, aplicado por el optimizador ---
    optimizador.step()

## Il tuo primo perceptrone multistrato

- Un perceptrone multistrato (MLP) sovrappone più livelli di neuroni. Il livello nascosto trasforma lo spazio di input fino a renderlo separabile
- Il problema XOR (or esclusivo) è l'esempio classico che un modello lineare non può risolvere, ma che si risolve con due neuroni in un livello nascosto
- Costruisci l'MLP con `torch.nn`, definisci una funzione di perdita e un ottimizzatore, e ripeti il ciclo di sempre: andata, `backward()`, `step()`
- Alla fine visualizzi la curva della perdita per verificare che la rete stia imparando

In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# Problema XOR: la salida es verdadera solo cuando exactamente una entrada vale 1
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.oculta = nn.Linear(2, 4)   # capa oculta: 2 entradas -> 4 neuronas
        self.salida = nn.Linear(4, 1)   # capa de salida: 4 -> 1

    def forward(self, x):
        h = torch.relu(self.oculta(x))
        return torch.sigmoid(self.salida(h))

modelo = MLP()
perdida_fn = nn.BCELoss()   # entropía cruzada binaria, para clasificación con 2 clases
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.05)

In [ ]:
historial_perdida = []

for epoca in range(500):
    y_pred = modelo(X)
    perdida = perdida_fn(y_pred, Y)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    historial_perdida.append(perdida.item())

print("Predicciones finales:")
for entrada, objetivo, pred in zip(X, Y, modelo(X)):
    print(f"{entrada.tolist()} -> objetivo={objetivo.item():.0f}  prediccion={pred.item():.3f}")

In [ ]:
plt.plot(historial_perdida)
plt.xlabel("Época")
plt.ylabel("Pérdida (BCE)")
plt.title("Curva de pérdida del MLP sobre XOR")
plt.show()

## Riepilogo della lezione

- Un neurone artificiale è una somma ponderata più una funzione di attivazione non lineare
- Sovrapporre livelli costruisce rappresentazioni gerarchiche: da pattern semplici a concetti astratti
- La *backpropagation* calcola il gradiente di ciascun peso applicando la regola della catena, all'indietro nel grafo computazionale
- PyTorch automatizza questo calcolo con *autograd*: `requires_grad=True`, `.backward()` e un ottimizzatore
- Un MLP con almeno un livello nascosto risolve problemi non lineari come XOR, impossibili per un singolo neurone
- Questo è il meccanismo di base che riutilizzano le CNN, le RNN, il Transformer, le GAN e i VAE che studierai nelle prossime lezioni

**Apri ora il notebook di pratica e metti in pratica ciò che hai appena visto: `../notebooks_practica/clase01_practica_intro_fundamentos_redes_neuronales.ipynb`**